# # Customer Lifetime Value Model
#
# This notebook-style script generates synthetic Day 0–7 user data, trains a
# separate D180 LTV model for each product, and compares campaign performance.
# Run the file from any working directory with `python notebooks/cel_ltv_model.py`.



# ## 1. Imports and project setup



In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

ROOT = Path("/Users/manupradeep/Downloads/CEL_end_to_end_reference")
DATA_DIR = ROOT / "data"
OUTPUT_DIR = ROOT / "outputs"
DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

rng = np.random.default_rng(42)



# ## 2. Generate synthetic user data

The synthetic data now includes Day-7 engagement, retention/recency, gameplay progression, social behaviour, and product-specific monetisation signals. CAC is retained only for downstream campaign economics/ROAS.


In [6]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)


def sigmoid(x):
    return 1 / (1 + np.exp(-x))


def make_data(n=12000):
    """
    Synthetic Day-0 to Day-7 dataset for the CEL Analyst Assignment.

    No game-specific metrics are used.
    CAC is a campaign-level economic measure, not an ML feature.
    """

    product = rng.choice(["subscription", "ad_supported"], n)
    campaign = rng.choice(["Campaign_A", "Campaign_B"], n)

    # Campaign-level CAC: used for ROAS, NOT as a model feature.
    campaign_cac = {"Campaign_A": 90.0, "Campaign_B": 105.0}
    cac = pd.Series(campaign).map(campaign_cac).to_numpy()

    # Hidden variable used only to generate realistic synthetic outcomes.
    # It is deliberately excluded from the model to avoid leakage.
    quality = rng.normal(0, 1, n)

    # -----------------------------
    # Engagement
    # -----------------------------
    active_days_7 = np.clip(
        np.rint(
            3.2
            + 0.9 * quality
            + np.where(campaign == "Campaign_A", -0.25, 0.25)
            + rng.normal(0, 1.0, n)
        ), 1, 7
    ).astype(int)

    sessions_7 = np.clip(
        np.rint(
            4.5
            + 2.0 * quality
            + 0.95 * active_days_7
            + np.where(campaign == "Campaign_A", 0.7, 0)
            + rng.normal(0, 2.4, n)
        ), 1, 40
    ).astype(int)

    avg_session_min_7 = np.clip(
        7 + 2.1 * quality + 0.55 * active_days_7
        + rng.normal(0, 2.4, n),
        2, 30
    )

    sessions_per_active_day_7 = sessions_7 / np.maximum(active_days_7, 1)

    # -----------------------------
    # Retention / recency
    # -----------------------------
    d1_retained = (
        rng.random(n)
        < sigmoid(-0.5 + 0.65 * quality + 0.18 * sessions_7)
    ).astype(int)

    d3_retained = (
        rng.random(n)
        < sigmoid(-1.1 + 0.55 * quality + 0.22 * active_days_7)
    ).astype(int)

    d7_retained = (active_days_7 >= 5).astype(int)

    last_active_day_7 = np.clip(
        np.rint(
            2.8
            + 1.9 * quality
            + 0.55 * active_days_7
            + rng.normal(0, 1.5, n)
        ), 0, 7
    ).astype(int)

    days_since_last_active_7 = 7 - last_active_day_7

    early_sessions = np.maximum(
        1, np.rint(sessions_7 * rng.uniform(0.30, 0.65, n)).astype(int)
    )
    late_sessions = np.maximum(0, sessions_7 - early_sessions)
    session_momentum_7 = late_sessions / np.maximum(early_sessions, 1)

    median_gap_hours_7 = np.clip(
        48
        - 4.0 * quality
        - 1.8 * sessions_per_active_day_7
        + rng.normal(0, 7, n),
        1, 96
    )

    # -----------------------------
    # Subscription monetisation
    # -----------------------------
    engagement_score = (
        0.40 * active_days_7 / 7
        + 0.35 * np.minimum(sessions_7 / 20, 1)
        + 0.25 * np.minimum(avg_session_min_7 / 20, 1)
    )

    trial_started_d7 = (
        (product == "subscription")
        & (
            rng.random(n)
            < sigmoid(
                -1.1
                + 2.3 * engagement_score
                + 0.75 * quality
                + np.where(campaign == "Campaign_A", -0.20, 0.20)
            )
        )
    ).astype(int)

    subscription_started_d7 = (
        (product == "subscription")
        & (
            rng.random(n)
            < sigmoid(
                -2.0
                + 2.8 * engagement_score
                + 1.15 * trial_started_d7
                + 0.65 * quality
            )
        )
    ).astype(int)

    days_to_trial = np.where(
        trial_started_d7 == 1,
        np.clip(
            np.rint(
                5 - 2.0 * quality
                - 0.6 * engagement_score
                + rng.normal(0, 1.2, n)
            ), 1, 7
        ),
        np.nan
    )

    # -----------------------------
    # Ad-supported monetisation
    # -----------------------------
    ad_impressions_7 = np.where(
        product == "ad_supported",
        np.clip(
            np.rint(
                5
                + 3.1 * sessions_7
                + 2.5 * quality
                + rng.normal(0, 7, n)
            ), 0, 220
        ),
        0
    ).astype(int)

    ad_views_7 = np.where(
        product == "ad_supported",
        np.clip(
            np.rint(0.72 * ad_impressions_7 + rng.normal(0, 4, n)),
            0, None
        ),
        0
    ).astype(int)

    rewarded_ads_watched_7 = np.where(
        product == "ad_supported",
        np.clip(
            np.rint(
                0.22 * ad_impressions_7
                + 2.0 * quality
                + rng.normal(0, 4, n)
            ), 0, None
        ),
        0
    ).astype(int)

    ad_view_rate_7 = ad_views_7 / np.maximum(ad_impressions_7, 1)
    rewarded_ad_rate_7 = (
        rewarded_ads_watched_7 / np.maximum(ad_impressions_7, 1)
    )

    # -----------------------------
    # General social / referral behaviour
    # -----------------------------
    social_actions_7 = np.clip(
        np.rint(
            0.3
            + 0.7 * quality
            + 0.10 * sessions_7
            + rng.normal(0, 1.2, n)
        ), 0, 20
    ).astype(int)

    # -----------------------------
    # Future target: D180 LTV
    # -----------------------------
    subscription_ltv = (
        20
        + 135 * subscription_started_d7
        + 35 * trial_started_d7
        + 24 * active_days_7
        + 2.8 * sessions_7
        + 7.0 * avg_session_min_7
        + 18 * d7_retained
        + 25 * quality
    )

    ad_supported_ltv = (
        10
        + 0.85 * ad_impressions_7
        + 0.38 * ad_views_7
        + 1.10 * rewarded_ads_watched_7
        + 16 * active_days_7
        + 2.6 * sessions_7
        + 4.5 * avg_session_min_7
        + 14 * d7_retained
        + 18 * quality
    )

    ltv = np.where(
        product == "subscription",
        subscription_ltv,
        ad_supported_ltv
    )

    # Campaign effect creates a useful D7-vs-D180 comparison scenario.
    ltv += np.where(campaign == "Campaign_A", 10, 30)

    noise_scale = np.where(product == "subscription", 38, 20)
    ltv = np.clip(
        ltv + rng.normal(0, noise_scale, n),
        0, None
    )

    return pd.DataFrame({
        "user_id": np.arange(1, n + 1),
        "product": product,
        "campaign": campaign,

        # Economics only; excluded from ML.
        "cac": cac,

        # General Day-7 features.
        "active_days_7": active_days_7,
        "sessions_7": sessions_7,
        "avg_session_min_7": avg_session_min_7,
        "sessions_per_active_day_7": sessions_per_active_day_7,

        # Retention / recency.
        "d1_retained": d1_retained,
        "d3_retained": d3_retained,
        "d7_retained": d7_retained,
        "last_active_day_7": last_active_day_7,
        "days_since_last_active_7": days_since_last_active_7,
        "session_momentum_7": session_momentum_7,
        "median_gap_hours_7": median_gap_hours_7,

        # Subscription-specific.
        "trial_started_d7": trial_started_d7,
        "subscription_started_d7": subscription_started_d7,
        "days_to_trial": days_to_trial,

        # Ad-supported-specific.
        "ad_impressions_7": ad_impressions_7,
        "ad_views_7": ad_views_7,
        "rewarded_ads_watched_7": rewarded_ads_watched_7,
        "ad_view_rate_7": ad_view_rate_7,
        "rewarded_ad_rate_7": rewarded_ad_rate_7,

        # General behavioural signal.
        "social_actions_7": social_actions_7,

        # Future target.
        "d180_ltv": ltv,
    })


def get_features(data, product):
    """
    Day-7 features used by the LTV models.
    CAC, campaign, user_id and D180 LTV are excluded.
    """

    common_features = [
        "active_days_7",
        "sessions_7",
        "avg_session_min_7",
        "sessions_per_active_day_7",
        "d1_retained",
        "d3_retained",
        "d7_retained",
        "days_since_last_active_7",
        "session_momentum_7",
        "median_gap_hours_7",
        "social_actions_7",
    ]

    if product == "subscription":
        product_features = [
            "trial_started_d7",
            "subscription_started_d7",
            "days_to_trial",
        ]
    else:
        product_features = [
            "ad_impressions_7",
            "ad_views_7",
            "rewarded_ads_watched_7",
            "ad_view_rate_7",
            "rewarded_ad_rate_7",
        ]

    return data[common_features + product_features]


df = make_data()
df.to_csv(DATA_DIR / "synthetic_users.csv", index=False)


# ## 3. Select product-specific model features

The model uses only signals observable by Day 7. Campaign and CAC are intentionally excluded from the LTV model and used after prediction for campaign-level ROAS.


In [3]:
def features(data, product):
    """Return Day-7 features for the selected product.

    CAC, campaign, user_id and D180 LTV are deliberately excluded.
    Campaign/CAC are used downstream for campaign economics and ROAS.
    """
    common_features = [
        "active_days_7",
        "sessions_7",
        "avg_session_min_7",
        "sessions_per_active_day_7",
        "d1_retained",
        "d3_retained",
        "d7_retained",
        "days_since_last_active_7",
        "session_momentum_7",
        "median_gap_hours_7",
        "social_actions_7",
    ]
    product_features = (
        ["trial_started_d7", "subscription_started_d7", "days_to_trial"]
        if product == "subscription"
        else [
            "ad_impressions_7",
            "ad_views_7",
            "rewarded_ads_watched_7",
            "ad_view_rate_7",
            "rewarded_ad_rate_7",
        ]
    )
    return data[common_features + product_features]


# ## 4. Train models and calculate validation metrics

Each product gets a mean-value baseline and a histogram gradient boosting model. Validation residuals provide a simple empirical uncertainty range. Campaign ROAS is calculated from total predicted revenue divided by total acquisition spend.


In [4]:
summary = []
metrics = []
associations = []

for product in ["subscription", "ad_supported"]:
    product_data = df[df["product"] == product].copy()
    X = features(product_data, product)
    y = product_data["d180_ltv"]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.25, random_state=42
    )

    baseline = DummyRegressor(strategy="mean").fit(X_train, y_train)
    baseline_predictions = baseline.predict(X_test)

    model = HistGradientBoostingRegressor(
        max_iter=250, learning_rate=0.05, max_leaf_nodes=15,
        l2_regularization=2, random_state=42
    ).fit(X_train, y_train)
    predictions = model.predict(X_test)

    metrics.extend([
        {"product": product, "model": "mean_baseline",
         "MAE": mean_absolute_error(y_test, baseline_predictions),
         "RMSE": mean_squared_error(y_test, baseline_predictions) ** 0.5,
         "R2": r2_score(y_test, baseline_predictions)},
        {"product": product, "model": "hist_gradient_boosting",
         "MAE": mean_absolute_error(y_test, predictions),
         "RMSE": mean_squared_error(y_test, predictions) ** 0.5,
         "R2": r2_score(y_test, predictions)},
    ])

    residual_quantiles = np.quantile(y_test.to_numpy() - predictions, [0.10, 0.90])
    product_data["predicted_d180_ltv"] = np.clip(model.predict(X), 0, None)
    product_data["lower_80"] = np.clip(product_data["predicted_d180_ltv"] + residual_quantiles[0], 0, None)
    product_data["upper_80"] = np.clip(product_data["predicted_d180_ltv"] + residual_quantiles[1], 0, None)

    for campaign in ["Campaign_A", "Campaign_B"]:
        campaign_data = product_data[product_data["campaign"] == campaign]

        if product == "subscription":
            value_proxy = (
                20 * campaign_data["trial_started_d7"].mean()
                + 50 * campaign_data["subscription_started_d7"].mean()
            )
        else:
            value_proxy = (
                0.20 * campaign_data["ad_impressions_7"].mean()
                + 0.35 * campaign_data["ad_views_7"].mean()
            )

        predicted_ltv = campaign_data["predicted_d180_ltv"].mean()
        actual_ltv = campaign_data["d180_ltv"].mean()
        campaign_cac = campaign_data["cac"].iloc[0]
        predicted_revenue = campaign_data["predicted_d180_ltv"].sum()
        actual_revenue = campaign_data["d180_ltv"].sum()
        acquisition_spend = len(campaign_data) * campaign_cac

        summary.append({
            "product": product,
            "campaign": campaign,
            "users": len(campaign_data),
            "campaign_cac": campaign_cac,
            "acquisition_spend": acquisition_spend,
            "d7_value_proxy": value_proxy,
            "predicted_d180_ltv": predicted_ltv,
            "predicted_ltv_lower_80": campaign_data["lower_80"].mean(),
            "predicted_ltv_upper_80": campaign_data["upper_80"].mean(),
            "actual_d180_ltv_synthetic": actual_ltv,
            "predicted_d180_roas": predicted_revenue / acquisition_spend,
            "actual_d180_roas_synthetic": actual_revenue / acquisition_spend,
        })

    for column in X.columns:
        associations.append({
            "product": product,
            "feature": column,
            "spearman_corr_with_d180_ltv": X[column].corr(y, method="spearman"),
        })


# ## 5. Export results

Exports the synthetic feature table, campaign economics, validation metrics, and feature associations.


In [5]:
summary_df = pd.DataFrame(summary)
metrics_df = pd.DataFrame(metrics)
associations_df = pd.DataFrame(associations)

summary_df.to_csv(OUTPUT_DIR / "campaign_summary.csv", index=False)
metrics_df.to_csv(OUTPUT_DIR / "validation_metrics.csv", index=False)
associations_df.sort_values(
    ["product", "spearman_corr_with_d180_ltv"], ascending=[True, False]
).to_csv(OUTPUT_DIR / "feature_associations.csv", index=False)

print(summary_df.round(2).to_string(index=False))
print("\nValidation\n", metrics_df.round(3).to_string(index=False))


     product   campaign  users  campaign_cac  acquisition_spend  d7_value_proxy  predicted_d180_ltv  predicted_ltv_lower_80  predicted_ltv_upper_80  actual_d180_ltv_synthetic  predicted_d180_roas  actual_d180_roas_synthetic
subscription Campaign_A   3004          90.0           270360.0           30.77              271.24                  220.54                  320.44                     263.32                 3.01                        2.93
subscription Campaign_B   3029         105.0           318045.0           35.09              301.78                  251.05                  350.99                     309.22                 2.87                        2.94
ad_supported Campaign_A   3010          90.0           270900.0           13.83              180.93                  152.00                  213.67                     173.78                 2.01                        1.93
ad_supported Campaign_B   2957         105.0           310485.0           13.35              192.65     